# D51 Gateway Deploy

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.10

# API 게이트웨이와 무중단 배포: 단일 진입점부터 버전 관리 · 배포 전략까지

**API 게이트웨이(API Gateway)는 여러 대의 모델 서빙 서버 앞단에서 모든 클라이언트 요청을 단일 주소로 받아 인증, 요청 제한, 부하 분산, 버전 전환을 처리하는 '중앙 안내 데스크'입니다.** 50일차까지 우리는 FastAPI 추론 서버와 Streamlit 화면을 도커 컨테이너로 묶어 띄웠습니다. 하지만 실제 서비스에서 사용자가 늘어나 추론 서버를 2대, 3대로 늘리거나 모델을 새 버전으로 교체하려 하면, 클라이언트가 어느 서버 주소로 요청을 보내야 할지 혼란이 생깁니다.

이번 차시에서는 교재 『LLMOps 완벽 가이드』 6장(6.5 API 게이트웨이, 6.6 API 버전 관리, 6.7 배포 아키텍처)의 핵심 개념을 직관적인 비유와 간결한 파이썬 코드로 실습한 뒤, **51일차 실습 부록인 `[프로젝트 1] 정형 데이터 예측 서비스`(`D52_Appendix_Housing.ipynb`)**로 이어집니다.

**이번 차시 학습목표**

1. 다중 서버 환경에서 API 게이트웨이의 6대 핵심 기능(라우팅, 부하 분산, 인증, 요청 제한, 로깅·추적, 캐싱)과 앞단 빠른 거절(Fail-Fast) 순서를 설명할 수 있다.
2. 요청 한도 초과(`429 Too Many Requests`)와 서버 용량 포화(`503 Service Unavailable`)의 반환 기준을 구분하고, 토큰 버킷(Token Bucket) 제한기와 서버 준비 상태(Readiness) 판정 로직을 파이썬으로 구현할 수 있다.
3. API 수명 주기 5단계와 버전 관리 원칙을 이해하고, 여유 자원 유무와 서비스 특성에 맞춰 블루-그린(Blue-Green), 카나리(Canary), 롤링(Rolling) 배포 전략을 선택·적용할 수 있다.

**오늘 다루지 않는 것**

- 상용 게이트웨이 제품(Nginx, Kong, AWS API Gateway)의 세부 설정 파일 문법 : 제품마다 문법은 다르지만 오늘 배우는 6가지 핵심 동작 원리는 모두 동일합니다
- 쿠버네티스(Kubernetes) 클러스터 내부의 인그레스(Ingress) 컨트롤러 운영 : 단일 진입점과 라우팅의 기초 개념에 집중합니다

## 실행 환경

**오늘 실습 코드는 외부 서버나 GPU 없이 파이썬 표준 라이브러리만으로 코랩(Colab)과 로컬 주피터에서 1초 만에 실행되도록 구성되어 있습니다.** 복잡한 백그라운드 프로세스를 띄우는 대신, 게이트웨이의 핵심 알고리즘(부하 분산, 토큰 버킷, 대기열 계산, 헬스체크 판정, 무중단 배포 라우팅)을 20줄 내외의 간결한 파이썬 함수로 직접 구현하고 눈으로 결과를 확인합니다.

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 내용 |
|---|---|
| 기준 환경 | Google Colab 또는 로컬 Jupyter Notebook (GPU 불필요, 추가 패키지 설치 불필요) |
| 연계 교재 | 『LLMOps 완벽 가이드』 6.5 (API 게이트웨이), 6.6 (API 버전 관리 및 수명 주기 관리), 6.7 (LLM 배포 아키텍처) |
| 연계 실습 부록 | 본 노트북 학습 후 **`D52_Appendix_Housing.ipynb` ([51일차 실습 부록 · 프로젝트 1] 정형 데이터 예측 서비스)** 진행 |

</div>

## 1. API 게이트웨이의 기초: 왜 서버 앞단에 단일 진입점이 필요한가

교재 : 6.5 API 게이트웨이, 6.7 LLM 배포 아키텍처

### 1.1 서버가 두 대 이상으로 늘어날 때 생기는 문제와 `nginx`의 진짜 역할

**추론 서버가 여러 대로 늘어났을 때 앞단에 API 게이트웨이가 없으면, 클라이언트(Streamlit 화면, 모바일 앱)가 모든 서버의 주소를 일일이 기억하고 어느 서버가 살아 있는지 직접 확인해야 합니다.**

대형 병원에 대표 안내 데스크가 없으면 환자가 진료실 1·2·3번 직통 번호를 모두 적어 두고 직접 확인하며 전화를 걸어야 합니다. API 게이트웨이는 서버들 앞단에 서는 **'대표 안내 데스크(단일 진입점, Single Point of Entry)'** 입니다.

> **50일차(`D50`) `nginx` 실습과의 연결**: Docker 실습 `2.5절`에서 우리가 컨테이너로 띄웠던 **`nginx`(엔진엑스)** 기억나시나요? `nginx`는 단순히 정적 웹페이지만 보여주는 도구가 아니라, 현업에서 여러 대의 FastAPI·vLLM 추론 컨테이너 앞단에 서서 손님의 요청을 대신 받아 뒤쪽 서버들로 똑똑하게 나눠 주는 **리버스 프록시(Reverse Proxy) 및 API 게이트웨이**로 가장 널리 쓰이는 핵심 인프라입니다.

![클라이언트는 게이트웨이 주소 하나만 알고 서버가 몇 대인지와 어느 서버가 대상에서 빠져 있는지는 게이트웨이가 압니다.](lms-img:0)

*클라이언트는 게이트웨이 주소 하나만 알고 서버가 몇 대인지와 어느 서버가 대상에서 빠져 있는지는 게이트웨이가 압니다.*

위 그림처럼 앞단에 게이트웨이를 세우면 두 가지가 즉시 단순해집니다.
- **클라이언트(부르는 쪽)는 게이트웨이 주소 하나만 알면 됩니다**: 뒤쪽 추론 서버가 2대에서 5대로 늘어나든, 서버 포트나 IP가 바뀌든 클라이언트 코드는 한 줄도 고칠 필요가 없습니다.
- **장애 서버 우회를 게이트웨이가 알아서 처리합니다**: 서버 한 대가 멈추면 게이트웨이가 그 서버를 목록에서 빼고 정상 동작 중인 다른 서버로 요청을 보냅니다.

### 1.2 교재 6.5가 정의하는 API 게이트웨이의 6대 핵심 기능

**교재 『LLMOps 완벽 가이드』 6.5절에서는 API 게이트웨이를 '모든 클라이언트 요청의 단일 진입점이자 백엔드 서비스로 요청을 전달하기 전 보안을 강화하고 성능을 최적화하는 관문'으로 정의하며 다음 6가지 핵심 역할을 제시합니다.**

<div style="font-size: 18px; line-height: 1.6;">

| 게이트웨이 6대 기능 (교재 6.5) | 쉬운 비유 | 게이트웨이가 없을 때 발생하는 문제 | 다루는 위치 |
|---|---|---|---|
| **1. 요청 라우팅 (Routing)** | 건물 1층 안내판 (`/v1`은 1층, `/chat`은 2층) | 클라이언트가 기능별·버전별 서버 주소를 각각 따로 호출해야 함 | 1.4절, 5절 |
| **2. 부하 분산 (Load Balancing)** | 은행 번호표 창구 배정 (빈 창구로 안내) | 특정 GPU 서버 한 대에만 요청이 몰려 응답이 느려지고 다른 서버는 놀게 됨 | 1.4절 |
| **3. 인증 및 권한 부여 (Auth)** | 입구 보안 검색대 (출입증 확인) | API Key가 없는 무자격 요청이 추론 서버까지 들어가 GPU를 소모함 | 2절 |
| **4. 요청 제한 (Rate Limiting)** | 1인당 구매 수량 제한 (사재기 방지) | 한 사용자가 초당 수백 건의 요청을 쏟아부어 다른 모든 사용자의 서비스가 멈춤 | 3절 |
| **5. 로깅 및 모니터링 (Logging)** | 출입 기록장 및 접수 번호(`Trace ID`) 발급 | 서버가 여러 대일 때 에러가 난 요청이 어느 서버에서 실패했는지 찾기 어려움 | 2.3절 |
| **6. 캐싱 (Caching)** | 자주 묻는 질문(FAQ) 즉답지 비치 | 똑같은 질문이 반복될 때마다 매번 비싼 LLM 추론을 다시 실행함 | 이론 정리 |

</div>

또한 교재 **6.7절(LLM 배포 아키텍처)** 에서는 모든 기능을 한 덩어리로 묶는 **모놀리식(Monolithic) 아키텍처**와, 전처리·추론·후처리를 독립된 컨테이너로 나누는 **마이크로서비스(Microservices) 아키텍처**를 비교합니다. 마이크로서비스 구조에서는 여러 컨테이너 앞단에 서서 요청을 조율하는 API 게이트웨이가 필수 구성 요소가 됩니다.

### 1.3 게이트웨이의 '빠른 거절(Fail-Fast)' 순서: 비용이 싼 검사부터 먼저 수행하라

**게이트웨이는 요청을 받았을 때 계산 비용이 가장 적게 드는 확인부터 순서대로 수행해, 잘못된 요청을 몇 밀리초(ms) 안에 돌려보내고 비싼 GPU 연산은 모든 관문을 통과한 요청에만 써야 합니다.**

![게이트웨이는 비용이 적은 확인부터 하므로 잘못된 요청은 서버에 닿기 전에 몇 밀리초 만에 거절됩니다.](lms-img:0)

*게이트웨이는 비용이 적은 확인부터 하므로 잘못된 요청은 서버에 닿기 전에 몇 밀리초 만에 거절됩니다.*

게이트웨이의 검문 순서는 항상 **① 인증(`401`) → ② 요청 한도(`429`) → ③ 대기열 여유 확인(`503`) → ④ 준비된 서버 중 가장 한가한 곳으로 전달** 순서입니다.

- 만약 **대기열(줄 세우기)**을 **인증(`401`)이나 요청 한도(`429`)**보다 먼저 수행하면 어떻게 될까요? API Key가 틀렸거나 이미 한도를 초과해 어차피 거절될 요청들이 대기열 자리를 꽉 채워, 정작 정상적인 자격을 갖춘 사용자의 요청이 `503(대기열 포화)`으로 쫓겨나는 사고가 발생합니다.
- 또한 48일차에서 배운 **실시간 스트리밍(`SSE`)** 응답을 중계할 때, 게이트웨이가 뒤쪽 서버의 답변이 끝날 때까지 다 모았다가(버퍼링) 한 번에 전달하면 **첫 글자가 나올 때까지의 체감 대기 시간(TTFT) 단축 효과와 연결 종료 시 즉시 생성 취소 기능 두 가지가 모두 사라집니다.** 따라서 게이트웨이는 서버가 보내는 청크를 받는 즉시 클라이언트로 흘려보내야 합니다.

### 1.4 [기초 실습] 라운드로빈(Round-Robin) vs 최소 연결(Least Connections) 부하 분산

**여러 대의 추론 서버로 요청을 골고루 나누는 부하 분산(Load Balancing) 방식 중 가장 대표적인 두 가지는 순서대로 번갈아 보내는 '라운드로빈'과 현재 처리 중인 요청이 가장 적은 서버로 보내는 '최소 연결'입니다.**

- **라운드로빈 (Round-Robin)**: 1번 서버 → 2번 서버 → 3번 서버 순서로 무조건 차례대로 돌아가며 배정합니다. 정형 데이터 예측(추론 시간 1ms 내외)처럼 **모든 요청의 처리 시간이 비슷할 때** 간단하고 효과적입니다.
- **최소 연결 (Least Connections)**: 현재 진행 중인 요청(활성 연결) 수가 가장 적은 서버를 골라 배정합니다. LLM 챗봇이나 텍스트 생성처럼 **질문마다 생성 길이가 달라 처리 시간이 0.5초~10초까지 크게 차이 날 때** 특정 서버에 긴 요청이 쌓이는 현상을 막아 줍니다.

실습 코드에 등장하는 핵심 파이썬 문법과 연산자의 역할은 다음과 같습니다.

<div style="font-size: 18px; line-height: 1.6;">

| 코드 표현 | 문법 구조 | 역할과 동작 원리 |
|---|---|---|
| **`step_idx % len(server_list)`** | 나머지 연산자 (`%`) | 요청 순번(`step_idx`)을 서버 대수(`3`)로 나눈 나머지를 구해 인덱스가 항상 `0 → 1 → 2 → 0 → 1 ...` 순서로 순환하게 만듭니다. |
| **`min(server_list, key=lambda s: s["active_requests"])`** | `min(반복객체, key=기준함수)` | 서버 리스트에서 값을 비교할 때 `lambda`(익명 함수)를 사용해 각 서버 딕셔너리의 **`"active_requests"`(현재 처리 중인 요청 수)가 가장 작은 서버**를 찾아 반환합니다. |

</div>

In [1]:
# 추론 서버 3대의 상태 (현재 처리 중인 요청 수 active_requests)
servers = [
    {"name": "gpu-server-1", "url": "http://10.0.0.1:8000", "active_requests": 2},
    {"name": "gpu-server-2", "url": "http://10.0.0.2:8000", "active_requests": 0},
    {"name": "gpu-server-3", "url": "http://10.0.0.3:8000", "active_requests": 1},
]

# 1. 라운드로빈 선택기: 서버의 현재 부하와 무관하게 순번(idx)대로 선택합니다
def pick_round_robin(server_list, step_idx):
    return server_list[step_idx % len(server_list)]

# 2. 최소 연결(Least Connections) 선택기: 현재 active_requests가 가장 적은 서버를 선택합니다
def pick_least_connections(server_list):
    return min(server_list, key=lambda s: s["active_requests"])

print("[현재 서버별 처리 중인 요청 수]")
for s in servers:
    print(f" - {s['name']}: {s['active_requests']}건 처리 중")

rr_choice = pick_round_robin(servers, step_idx=0)
lc_choice = pick_least_connections(servers)

print(f"\n👉 라운드로빈(0번 차례) 선택 결과 : {rr_choice['name']} (이미 {rr_choice['active_requests']}건 처리 중인데도 배정됨)")
print(f"👉 최소 연결(가장 한가한 곳) 선택 : {lc_choice['name']} (현재 {lc_choice['active_requests']}건 처리 중인 유휴 서버에 배정됨)")

[현재 서버별 처리 중인 요청 수]
 - gpu-server-1: 2건 처리 중
 - gpu-server-2: 0건 처리 중
 - gpu-server-3: 1건 처리 중

👉 라운드로빈(0번 차례) 선택 결과 : gpu-server-1 (이미 2건 처리 중인데도 배정됨)
👉 최소 연결(가장 한가한 곳) 선택 : gpu-server-2 (현재 0건 처리 중인 유휴 서버에 배정됨)


### 체크포인트

1. 추론 서버가 여러 대로 늘어났을 때 앞단에 API 게이트웨이를 두면 클라이언트 입장에서 얻을 수 있는 가장 큰 이점은 무엇입니까? (목표 1)
2. 게이트웨이가 요청을 대기열에 줄 세우기 전에 API Key 인증(`401`)과 요청 한도 검사(`429`)를 먼저 수행해야 하는 이유는 무엇입니까? (목표 1)
3. 모든 요청의 추론 시간이 1ms로 일정한 정형 데이터 모델과 달리, 답변 길이에 따라 추론 시간이 1초~10초까지 크게 달라지는 LLM 서빙에서 라운드로빈 대신 '최소 연결(Least Connections)' 부하 분산을 권장하는 이유는 무엇입니까? (목표 1)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. 클라이언트는 뒤쪽 서버가 몇 대인지, 각 서버의 IP·포트가 무엇인지, 어느 서버가 점검 중인지 몰라도 **게이트웨이의 단일 주소 하나만 호출**하면 됩니다.
2. 자격이 없거나 한도를 초과해 어차피 거절될 요청을 대기열에 넣으면 정상 사용자의 대기열 자리를 빼앗아 불필요한 `503` 거절을 유발하므로, 가장 비용이 싼 인증·한도 검사를 맨 앞에서 먼저 수행해 즉시 돌려보내야 합니다.
3. 라운드로빈은 번호 순서대로만 배정하므로 우연히 긴 답변 생성 요청이 연속으로 배정된 서버에 과부하가 걸릴 수 있습니다. 반면 최소 연결 방식은 **현재 처리 중인 요청 수가 가장 적은 한가한 서버로 새 요청을 보내므로** GPU 활용률을 고르게 유지할 수 있습니다.

</details>

## 2. 앞단 통합 인증(401)과 분산 로그 추적(Trace ID)

## 2. 앞단 통합 인증(`401`)과 분산 로그 추적(`Trace ID`)

교재 : 6.4 자격 증명 관리, 6.5 API 게이트웨이

### 2.1 인증의 일원화와 50일차 Docker Compose '뒷문(우회 접근)' 차단 원칙

**API Key 검사나 OAuth/JWT 토큰 검증을 게이트웨이 한 곳에 모아 두면, 자격 없는 요청(`401 Unauthorized`)이 추론 서버에 닿기 전에 차단되어 GPU 자원을 보호하고 키 목록도 한 곳에서만 관리할 수 있습니다.**

그런데 이때 반드시 함께 지켜야 할 보안 원칙이 하나 있습니다: **게이트웨이에서 인증을 수행하더라도, 뒤쪽 백엔드 추론 컨테이너의 포트가 호스트 외부에 그대로 열려 있으면 주소를 아는 사람이 게이트웨이를 건너뛰고 백엔드 서버를 직접 호출할 수 있습니다.**

![게이트웨이 컨테이너만 호스트 포트(127.0.0.1:8000)를 열고 뒤쪽 추론 컨테이너(api-1, api-2)는 내부망 전용으로 두어 우회 접근(뒷문)을 원천 차단합니다.](lms-img:0)

*게이트웨이 컨테이너만 호스트 포트(127.0.0.1:8000)를 열고 뒤쪽 추론 컨테이너(api-1, api-2)는 내부망 전용으로 두어 우회 접근(뒷문)을 원천 차단합니다.*

50일차에서 배운 `compose.yaml`로 이 구조를 구현할 때는 아래처럼 **앞단 `gateway` 서비스에만 `ports: ["127.0.0.1:8000:8000"]`를 열어 두고, 뒤쪽 `api-1`, `api-2` 추론 컨테이너에는 `ports:` 설정 자체를 적지 않아** 오직 Compose 내부 네트워크(`http://api-1:8000`, `http://api-2:8000`)로만 접근할 수 있게 격리합니다.

```yaml
# [50~51일차 연결 예시] 게이트웨이만 외부 포트를 열고 추론 컨테이너는 내부망으로 숨긴 compose.yaml 구조
services:
  gateway:
    build: {context: ., dockerfile: Dockerfile.gateway}
    ports:
      - "127.0.0.1:8000:8000"   # 유일한 외부 진입점 (여기서 401 인증 · 429 한도 · Trace ID 처리)
    depends_on: [api-1, api-2]

  api-1:
    build: {context: ., dockerfile: Dockerfile}
    # ports 블록을 아예 적지 않음! -> 외부에서 뒷문 접근 불가, 게이트웨이만 http://api-1:8000으로 호출 가능
    volumes: ["./models:/app/models:ro"]

  api-2:
    build: {context: ., dockerfile: Dockerfile}
    # ports 블록 없음! -> 동일한 가중치 볼륨(:ro)을 공유하는 두 번째 추론 서버
    volumes: ["./models:/app/models:ro"]
```

### 2.2 분산 환경의 사건 접수 번호: `Trace ID` (`X-Request-ID`)

**추론 서버가 여러 대일 때 특정 사용자의 요청에서 에러가 발생하면 어느 서버의 로그를 봐야 할지 알기 어려우므로, 게이트웨이가 요청이 들어오는 첫 순간에 고유 번호(`Trace ID`)를 발급해 요청 헤더(`X-Request-ID`)와 응답 헤더에 함께 실어 보냅니다.**

택배를 보낼 때 접수처(게이트웨이)에서 **운송장 번호(`Trace ID`)**를 하나 붙여 주면, 그 상자가 어느 물류센터(추론 서버 1, 2, 3)를 거쳤는지 번호 하나로 끝까지 추적할 수 있는 것과 같습니다.

<div style="font-size: 18px; line-height: 1.6;">

| 헤더 / 파이썬 함수 | 정체와 문법 구조 | 역할과 동작 원리 |
|---|---|---|
| **`X-Request-ID` 헤더** | HTTP 표준 관례 추적 헤더 | 요청의 맨 앞단(게이트웨이)에서 부여해 뒤쪽 모든 마이크로서비스와 로그 파일에 전달하는 **사건 접수 번호**입니다. |
| **`uuid.uuid4().hex[:8]`** | 파이썬 표준 `uuid` 모듈 | 전 세계에서 겹치지 않는 난수 기반 고유 식별자(`UUID v4`)를 생성한 뒤, 하이픈이 제거된 32자리 16진수 문자열(`.hex`)에서 앞 8자리(`[:8]`)만 잘라 읽기 편한 추적 번호(예: `req-3f9a1c02`)를 만듭니다. |
| **`client_trace_id or f"req-..."`** | 파이썬 단락 평가 (`or`) | 클라이언트가 이미 `X-Request-ID`를 붙여서 보냈다면 그 번호를 그대로 유지하고, 없을 때(`None` 또는 `""`)만 새 번호를 발급합니다. |

</div>

In [2]:
import uuid

VALID_API_KEYS = {
    "key-hr-portal": "인사팀_포털",
    "key-sales-bot": "영업팀_챗봇",
}

def gateway_handle_request(path: str, api_key: str | None, client_trace_id: str | None = None):
    """게이트웨이의 1단계(Trace ID 발급)와 2단계(API Key 인증)를 수행합니다."""
    # 1. 클라이언트가 보낸 추적 번호가 있으면 쓰고, 없으면 새로 발급합니다
    trace_id = client_trace_id or f"req-{uuid.uuid4().hex[:8]}"

    # 2. API Key 인증 검사 (통과 못 하면 백엔드 서버를 부르지 않고 즉시 401 반환)
    if not api_key or api_key not in VALID_API_KEYS:
        print(f"[GW LOG] trace={trace_id} | status=401 | reason=인증 실패 (백엔드 미호출)")
        return {"status": 401, "trace_id": trace_id, "detail": "유효한 X-API-Key가 필요합니다."}

    caller = VALID_API_KEYS[api_key]
    print(f"[GW LOG] trace={trace_id} | caller={caller} | path={path} -> 백엔드 서버로 전달")
    # 백엔드 서버에도 같은 trace_id를 넘겨 서버 로그에 남기게 합니다
    print(f"  [SERVER LOG] trace={trace_id} | 추론 완료 (200 OK)")
    return {"status": 200, "trace_id": trace_id, "caller": caller, "result": "추론 응답 데이터"}

print("--- 테스트 1: API Key 없이 호출 ---")
print(gateway_handle_request("/v1/predict", api_key=None))

print("\n--- 테스트 2: 올바른 API Key로 호출 ---")
print(gateway_handle_request("/v1/predict", api_key="key-hr-portal"))

--- 테스트 1: API Key 없이 호출 ---
[GW LOG] trace=req-9c84f618 | status=401 | reason=인증 실패 (백엔드 미호출)
{'status': 401, 'trace_id': 'req-9c84f618', 'detail': '유효한 X-API-Key가 필요합니다.'}

--- 테스트 2: 올바른 API Key로 호출 ---
[GW LOG] trace=req-e30c352e | caller=인사팀_포털 | path=/v1/predict -> 백엔드 서버로 전달
  [SERVER LOG] trace=req-e30c352e | 추론 완료 (200 OK)
{'status': 200, 'trace_id': 'req-e30c352e', 'caller': '인사팀_포털', 'result': '추론 응답 데이터'}


### 체크포인트

1. 게이트웨이에 API Key 인증을 적용한 뒤 백엔드 서버의 인증 코드를 제거하려 할 때, 반드시 먼저 확인해야 하는 네트워크 조건은 무엇입니까? (목표 1)
2. 백엔드 서버가 여러 대일 때 개별 서버가 아닌 앞단 게이트웨이가 `Trace ID`를 발급해야 하는 이유는 무엇입니까? (목표 1)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. 외부 클라이언트가 게이트웨이를 우회해 백엔드 서버를 직접 호출할 수 없도록 **백엔드 서버의 포트를 외부 망에서 닫고 오직 게이트웨이만 접근할 수 있는 내부망으로 격리**되어 있는지 먼저 확인해야 합니다.
2. 클라이언트는 게이트웨이와만 통신하며 어떤 요청이 몇 번 서버로 갔는지(또는 서버에 닿기 전에 거절되었는지)는 게이트웨이만 알고 있으므로, **진입점인 게이트웨이에서 고유 번호(`Trace ID`)를 발급해 서버로 전달해야 게이트웨이 로그와 백엔드 서버 로그를 하나로 연결**할 수 있습니다.

</details>

## 3. 요청 제한(429)과 대기열(503) 제어의 기초

## 3. 요청 제한(`429`)과 대기열(`503`) 제어의 기초

교재 : 6.5 API 게이트웨이

### 3.1 요청 제한(Rate Limiting): 한 사용자의 독점을 막는 안전장치

**요청 제한(Rate Limiting)은 특정 클라이언트(사용자 또는 시스템)가 정해진 시간 동안 보낼 수 있는 요청 횟수에 상한선을 걸어, 한 곳에서 몰린 대량 요청이 전체 서버의 GPU 슬롯을 독점하지 못하게 막는 기능입니다(교재 6.5).**

예를 들어 영업팀 누군가가 `for` 반복문으로 1초에 100건의 요약 요청을 보내는 스크립트를 돌렸다고 가정해 보겠습니다. 요청 제한이 없으면 이 100건이 서버 슬롯을 전부 차지해 다른 직원의 실시간 질문이 모두 멈춥니다. 하지만 호출자별로 **"초당 최대 2건, 순간 최대 5건"** 처럼 한도를 걸어 두면, 한도를 넘긴 영업팀 스크립트에게만 **`429 Too Many Requests`** 와 함께 몇 초 뒤에 다시 시도하라는 **`Retry-After`** 헤더를 돌려보내고 다른 직원의 요청은 평소처럼 즉시 처리합니다.

### 3.2 고정 윈도우(Fixed Window)의 한계와 토큰 버킷(Token Bucket) 원리

**요청 횟수를 세는 가장 단순한 방법인 '고정 윈도우(예: 매 10초 칸마다 최대 10건)'는 구간의 경계(9.9초에 10건 + 10.1초에 10건)에서 순간적으로 한도의 2배(20건)가 쏟아지는 문제를 막지 못하므로, 실무 게이트웨이는 대부분 '토큰 버킷(Token Bucket)' 알고리즘을 사용합니다.**

![같은 "10초에 10건"이라도 고정 윈도우는 구간 경계에서 한도의 두 배를 통과시키고 토큰 버킷은 몰린 요청을 버킷 크기만큼만 받습니다.](lms-img:0)

*같은 "10초에 10건"이라도 고정 윈도우는 구간 경계에서 한도의 두 배를 통과시키고 토큰 버킷은 몰린 요청을 버킷 크기만큼만 받습니다.*

**토큰 버킷(Token Bucket)**은 이름 그대로 **'통(버킷) 안에 든 코인(토큰)'** 으로 생각하면 아주 쉽습니다.

<div style="font-size: 18px; line-height: 1.6;">

| 구성 요소 / 응답 헤더 | 의미와 비유 | 동작 원리 및 계산식 |
|---|---|---|
| **버킷 크기 (`capacity`)** | 통에 담을 수 있는 **최대 코인 개수** | 평소 코인이 꽉 차 있을 때 **한꺼번에 몰려서 통과할 수 있는 순간 최대 요청 수(버스트, Burst 허용량)**를 결정합니다. |
| **초당 충전 속도 (`refill_per_sec`)** | 1초마다 통에 새로 떨어지는 코인 개수 | 장기적인 **평균 요청 허용 속도**를 결정합니다 (`경과 시간 × refill_per_sec`만큼 코인 충전, 단 `capacity`를 초과하지 않음). |
| **`429 Too Many Requests`** | 요청 한도 초과 상태 코드 | 통에 남은 코인이 `1.0`개 미만일 때 반환하는 HTTP 상태 코드입니다(특정 호출자 1명만 차단하며 다른 사용자는 정상 이용). |
| **`Retry-After` 응답 헤더** | 재시도 대기 시간(초) 안내 | 코인이 다시 1개 모일 때까지 걸리는 시간인 **`(1.0 - 현재 코인 수) ÷ 초당 충전 속도`** 초를 계산해 클라이언트에게 *"이 시간만큼 쉬었다가 다시 보내세요"*라고 알려줍니다. |

</div>

먼저 `TokenBucket` 클래스를 정의한 뒤, 다음 셀에서 동시 요청 시나리오를 실행해 봅니다.

In [3]:
class TokenBucket:
    """호출자별 요청 한도를 제어하는 토큰 버킷 클래스입니다."""
    def __init__(self, capacity: int, refill_per_sec: float):
        self.capacity = capacity              # 최대 버킷 크기 (순간 버스트 허용량)
        self.refill_per_sec = refill_per_sec  # 초당 충전되는 토큰 수 (평균 허용 속도)
        self.tokens = float(capacity)         # 현재 남아 있는 토큰 수
        self.last_time = 0.0

    def allow_request(self, now_sec: float) -> tuple[bool, float]:
        """now_sec 시각에 들어온 요청의 통과 여부(bool)와 대기 필요 시간(retry_after)을 반환합니다."""
        # 1. 지난 시간만큼 토큰을 충전하되 버킷 크기(capacity)를 넘지 않게 합니다
        elapsed = max(0.0, now_sec - self.last_time)
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_per_sec)
        self.last_time = now_sec

        # 2. 토큰이 1개 이상 있으면 1개를 차감하고 통과시킵니다
        if self.tokens >= 1.0:
            self.tokens -= 1.0
            return True, 0.0

        # 3. 토큰이 부족하면 1개가 모일 때까지 기다려야 할 시간(Retry-After)을 계산해 거절합니다
        retry_after = round((1.0 - self.tokens) / self.refill_per_sec, 2)
        return False, retry_after

print("TokenBucket 클래스 정의 완료")

TokenBucket 클래스 정의 완료


In [4]:
# [시뮬레이션] 버킷 크기 3개, 초당 1개 충전되는 제한기에 0.0초 4건 연속 요청 후 1.0초 뒤 1건 요청
limiter = TokenBucket(capacity=3, refill_per_sec=1.0)

for req_no, t in enumerate([0.0, 0.0, 0.0, 0.0, 1.0], start=1):
    allowed, retry = limiter.allow_request(now_sec=t)
    status = "200 통과" if allowed else f"429 거절 (Retry-After: {retry}초)"
    print(f"요청 #{req_no} (시각 {t:3.1f}초) -> {status} | 남은 토큰: {limiter.tokens:.1f}개")

요청 #1 (시각 0.0초) -> 200 통과 | 남은 토큰: 2.0개
요청 #2 (시각 0.0초) -> 200 통과 | 남은 토큰: 1.0개
요청 #3 (시각 0.0초) -> 200 통과 | 남은 토큰: 0.0개
요청 #4 (시각 0.0초) -> 429 거절 (Retry-After: 1.0초) | 남은 토큰: 0.0개
요청 #5 (시각 1.0초) -> 200 통과 | 남은 토큰: 0.0개


### 3.3 대기열(Queue) 길이 산정과 `429` vs `503` 상태 코드 구분

**개별 호출자의 한도는 넘지 않았더라도 모든 사용자가 동시에 몰려 서버의 처리 슬롯이 꽉 찼을 때는 게이트웨이 대기열(Queue)에 줄을 세우며, 클라이언트가 기다려 줄 제한 시간(Timeout) 내에 차례가 오지 못할 요청은 즉시 `503 Service Unavailable`로 거절해야 합니다.**

![게이트웨이는 기다려 줄 시간 안에 답변을 받을 수 있는 요청까지만 대기열에 넣고 나머지에는 바로 다시 보낼 시점을 알립니다.](lms-img:0)

*게이트웨이는 기다려 줄 시간 안에 답변을 받을 수 있는 요청까지만 대기열에 넣고 나머지에는 바로 다시 보낼 시점을 알립니다.*

#### 1) 왜 대기열을 무한정 길게 늘리면 안 될까요?
대기열을 무작정 길게 늘리면 뒷사람은 클라이언트 제한 시간(Timeout)이 지나 연결을 끊고 떠나는데, 서버는 이미 떠난 사람 몫의 답변을 뒤늦게 생성하느라 GPU만 낭비합니다.

따라서 **최대 대기열 길이(`max_queue_size`)**는 클라이언트가 기다려 줄 시간(`timeout_sec`) 안에서 답변을 완료할 수 있는 인원까지만 허용합니다.

$$\text{최대 대기열 길이} = \text{전체 동시 처리 슬롯 수} \times \left( \left\lfloor \frac{\text{클라이언트 제한 시간}}{\text{1건 평균 추론 시간}} \right\rfloor - 1 \right)$$

예를 들어 전체 슬롯 **4개**, 1건 추론 **2초**, 제한 시간 **6초**라면 최대 대기열 길이는 $4 \times (\lfloor 6 \div 2 \rfloor - 1) = 8$건이며, 9번째부터는 즉시 `503`으로 거절합니다.

#### 2) 현업에서 가장 자주 묻는 `429`와 `503`의 차이

<div style="font-size: 18px; line-height: 1.6;">

| 구분 | `429 Too Many Requests` (요청 제한) | `503 Service Unavailable` (대기열 포화 / 미준비) |
|---|---|---|
| **누구의 사정인가** | **특정 클라이언트(호출자 1명)**가 자기에게 허용된 요청 한도를 초과함 (`4xx` 클라이언트 측 원인) | **서버·게이트웨이 전체**의 슬롯과 대기열이 꽉 찼거나 준비된 서버가 없음 (`5xx` 서버 측 원인) |
| **다른 사용자에게 미치는 영향** | 다른 정상 사용자의 요청은 아무 지장 없이 `200 OK`로 처리됨 | 현재 서버 전체가 바빠서 누구의 새 요청이든 수용할 수 없음 |
| **클라이언트의 올바른 대응** | `4xx` 중 유일하게 재시도 가능 → **`Retry-After` 초만큼 기다린 뒤 재시도** | **`Retry-After` 초만큼 기다린 뒤 재시도** |

</div>

아래 코드에서는 파이썬의 정수 나눗셈(몫) 연산자인 `//`(`int(client_timeout_sec // infer_sec)`)를 사용해 제한 시간 안에 몇 회차의 추론이 돌아갈 수 있는지 소수점 이하를 버리고 계산합니다.

In [5]:
# 실습: 클라이언트 타임아웃 안에 처리 가능한 적정 대기열 길이 계산 함수
def calc_max_queue_size(total_slots: int, infer_sec: float, client_timeout_sec: float) -> int:
    """클라이언트 타임아웃 내에 응답을 마칠 수 있는 최대 대기열 길이를 계산합니다."""
    batches_within_timeout = int(client_timeout_sec // infer_sec)
    waiting_batches = max(0, batches_within_timeout - 1)
    return total_slots * waiting_batches

# 예시 1: 서버 2대 × 슬롯 2개 = 총 4슬롯, 추론 2초, 타임아웃 6초
q1 = calc_max_queue_size(total_slots=4, infer_sec=2.0, client_timeout_sec=6.0)
# 예시 2: 서버 4대 × 슬롯 2개 = 총 8슬롯, 추론 3초, 타임아웃 10초
q2 = calc_max_queue_size(total_slots=8, infer_sec=3.0, client_timeout_sec=10.0)

print(f"예시 1 (슬롯 4개, 추론 2초, 제한시간 6초)  -> 적정 최대 대기열: {q1}건")
print(f"예시 2 (슬롯 8개, 추론 3초, 제한시간 10초) -> 적정 최대 대기열: {q2}건")

예시 1 (슬롯 4개, 추론 2초, 제한시간 6초)  -> 적정 최대 대기열: 8건
예시 2 (슬롯 8개, 추론 3초, 제한시간 10초) -> 적정 최대 대기열: 16건


### 체크포인트

1. 토큰 버킷(Token Bucket) 알고리즘에서 **버킷 크기(`capacity`)**와 **충전 속도(`refill_per_sec`)** 는 각각 무엇을 제어합니까? (목표 2)
2. 게이트웨이가 `429 Too Many Requests`를 반환할 때와 `503 Service Unavailable`을 반환할 때를 구분하는 핵심 기준은 무엇입니까? (목표 2)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. **버킷 크기(`capacity`)** 는 한꺼번에 몰려서 들어올 때 즉시 통과시킬 수 있는 **순간 최대 허용량(Burst)** 을 제어하고, **충전 속도(`refill_per_sec`)** 는 장기적으로 유지되는 **평균 요청 처리 속도** 를 제어합니다.
2. **`429`**는 서버 전체 용량과 무관하게 **특정 호출자가 자신에게 부여된 요청 한도를 넘었을 때** 반환하고, **`503`** 은 **서버 전체의 처리 슬롯과 대기열이 모두 찼거나 준비된 백엔드 서버가 하나도 없을 때** 반환합니다.

</details>

## 4. 생존 확인(Liveness)과 준비 상태(Readiness) 헬스체크

교재 : 6.1.5 5단계: 모니터링 설정하기, 6.5 API 게이트웨이

### 4.1 "살아 있음(Liveness)"과 "요청을 받을 준비가 됨(Readiness)"은 다르다

**수 기가바이트짜리 LLM이나 딥러닝 모델 서버는 프로세스가 켜진 뒤에도 모델 가중치를 GPU 메모리에 다 올리기까지 수십 초가 걸리므로, 게이트웨이는 서버가 단순히 살아 있는지(Liveness)가 아니라 '모델 로딩이 끝나 추론할 준비가 되었는지(Readiness)'를 확인한 뒤에야 요청을 보내야 합니다.**

식당에 불이 켜진 상태(`Liveness`)와 재료 손질이 끝나 손님을 받을 수 있는 영업 준비 완료 상태(`Readiness`)를 구분하는 것과 같습니다.

![게이트웨이는 살아 있는 서버라도 받을 준비가 끝난 서버에만 요청을 보냅니다.](lms-img:0)

*게이트웨이는 살아 있는 서버라도 받을 준비가 끝난 서버에만 요청을 보냅니다.*

<div style="font-size: 18px; line-height: 1.6;">

| 서버의 `/health` 응답 상태 | 살아 있음 (Liveness) | 받을 준비 (Readiness) | 게이트웨이 및 인프라의 조치 |
|---|---|---|---|
| 연결 거부 / 타임아웃 (응답 없음) | ❌ 죽어 있음 | ❌ 안 됨 | 라우팅 대상에서 즉시 제외하고 컨테이너 재기동 |
| `200 OK` + `{"status": "loading"}` | ✅ 살아 있음 | ❌ **아직 모델 로딩 중** | 죽이지 않고 기다리되, **`healthy`가 될 때까지 추론 요청은 보내지 않음** |
| `200 OK` + `{"status": "healthy"}` | ✅ 살아 있음 | ✅ **준비 완료** | 정상 라우팅 대상에 포함해 추론 요청 전달 |

</div>

### 4.2 `/health` 엔드포인트에 절대 넣지 말아야 할 두 가지

**헬스체크(`/health`)는 게이트웨이가 수 초마다 반복해서 호출하는 경로이므로, 그 안에 '실제 모델 추론'이나 '순간의 동시 슬롯 사용량'을 넣으면 오히려 멀쩡한 서버들이 도미노처럼 전멸하는 장애가 발생합니다.**

1. **절대 넣지 말 것 1 — 실제 LLM 토큰 생성(추론)**: "모델이 진짜로 잘 도는지 보자"며 `/health`가 호출될 때마다 짧은 문장을 생성하게 만들면, 피크 시간대에 헬스체크가 실제 사용자 질문과 GPU 슬롯을 두고 경쟁합니다. 결국 슬롯을 기다리느라 `/health` 응답이 2초 제한 시간을 넘기면 게이트웨이가 멀쩡한 서버를 '죽었다'고 판단해 목록에서 빼 버립니다. (모델 동작 검증은 **서버 기동 시 `lifespan`에서 딱 한 번**만 수행하고, `/health`는 준비 완료 플래그 변수만 즉시 반환해야 합니다.)
2. **절대 넣지 말 것 2 — 순간의 슬롯 포화 상태**: 현재 처리 슬롯 2개가 다 찼다고 `/health`가 `unhealthy`를 반환하게 만들면, 조금만 바빠져도 서버가 라우팅 목록에서 빠지고 그 트래픽이 남은 서버로 몰려 남은 서버들까지 연쇄적으로 빠지는 도미노 장애가 일어납니다.

아래 코드 셀을 실행해 올바른 Readiness 판정 함수를 작성하고, 모델 로딩 중인 서버를 라우팅에서 안전하게 걸러내는 동작을 확인해 보십시오.

In [6]:
def is_server_ready(http_status: int, body: dict) -> bool:
    """/health 응답의 상태 코드가 200이고 status 필드가 'healthy'일 때만 준비 완료(True)로 판정합니다."""
    if http_status != 200 or not isinstance(body, dict):
        return False
    return body.get("status") == "healthy"

# 3대의 서버에 대한 헬스체크 응답 시뮬레이션
health_responses = [
    {"name": "server-1 (정상 운영 중)", "http_status": 200, "body": {"status": "healthy", "version": "1.0.0"}},
    {"name": "server-2 (모델 로딩 중)", "http_status": 200, "body": {"status": "loading", "version": "1.1.0"}},
    {"name": "server-3 (응답 오류)",    "http_status": 500, "body": {"status": "error"}},
]

ready_servers = []
for s in health_responses:
    ready = is_server_ready(s["http_status"], s["body"])
    print(f"{s['name']:24} -> 준비 완료 여부: {ready}")
    if ready:
        ready_servers.append(s["name"])

print(f"\n✅ 게이트웨이가 실제 추론 요청을 보낼 대상 서버: {ready_servers}")

server-1 (정상 운영 중)       -> 준비 완료 여부: True
server-2 (모델 로딩 중)       -> 준비 완료 여부: False
server-3 (응답 오류)         -> 준비 완료 여부: False

✅ 게이트웨이가 실제 추론 요청을 보낼 대상 서버: ['server-1 (정상 운영 중)']


### 체크포인트

1. 프로세스는 켜져 있지만 아직 GPU 메모리에 모델을 올리고 있는 서버는 `/health`에서 어떤 상태를 반환해야 하며, 게이트웨이는 이 서버를 어떻게 다루어야 합니까? (목표 2)
2. `/health` 엔드포인트 안에서 매번 1토큰짜리 실제 LLM 추론을 실행하도록 만들면 바쁜 시간대에 어떤 문제가 발생합니까? (목표 2)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. 프로세스가 살아 있음을 알리는 HTTP `200`(또는 별도 Readiness 상태)과 함께 `"status": "loading"`을 반환해야 하며, 게이트웨이는 이 서버를 재시작하지 않고 기다리되 **`"status": "healthy"`가 될 때까지 추론 요청 배정 대상에서는 제외**해야 합니다.
2. 헬스체크가 실제 사용자 요청과 GPU 슬롯을 놓고 경쟁하게 되어, 바쁜 시간대에 헬스체크 응답이 타임아웃(예: 2초)을 넘기면서 **멀쩡한 서버가 라우팅에서 제외되고 남은 서버로 부하가 쏠려 전체 서버가 연쇄적으로 빠지는 장애**가 발생합니다.

</details>

## 5. API 버전 관리와 3대 무중단 배포 전략 (블루-그린 · 카나리 · 롤링)

교재 : 6.6 API 버전 관리 및 수명 주기 관리

### 5.1 API 버전(`/v1`, `/v2`)과 API 수명 주기 5단계 (교재 6.6)

**모델 가중치나 프롬프트만 개선되어 입출력 JSON 형식이 그대로라면 서버 내부 버전(`1.0.0` → `1.1.0`)만 올리면 되지만, 요청·응답 필드 이름이 바뀌어 기존 클라이언트 코드가 깨지는 변경이라면 반드시 새 API 버전(`/v2/predict`)을 열고 구버전(`/v1/predict`)을 단계적으로 폐기해야 합니다.**

교재 **6.6절(API 버전 관리 및 수명 주기 관리)** 에서는 API의 일생을 다음 5단계로 정리합니다.

<div style="font-size: 18px; line-height: 1.6;">

| 수명 주기 단계 (교재 6.6) | 핵심 활동 | 게이트웨이에서의 처리 |
|---|---|---|
| **1. 기획 (Planning)** | 비즈니스 요구사항 정의, 입출력 스키마 및 버전 규칙(`/v1`, 헤더 버전) 설계 | 라우팅 경로 설계 |
| **2. 개발 (Development)** | FastAPI 엔드포인트 구현, 입출력 검증(`Pydantic`), 인증·보안 연동 | 개발 서버 라우팅 연결 |
| **3. 테스트 (Testing)** | 기능 테스트, 비정상 입력(`422`)·인증(`401`) 방어 테스트, 부하 테스트 | 테스트 트래픽 검증 |
| **4. 배포 (Deployment)** | 운영 환경에 새 버전 서버 투입 및 트래픽 무중단 전환 | **블루-그린 · 카나리 · 롤링 가중치 조절** |
| **5. 폐기 (Deprecation)** | 구버전(`/v1`) 사용자에게 종료 일정 사전 공지 후 안전하게 닫기 | `/v1` 응답에 **`Deprecation` 예고 헤더** 부착 → 로그로 미이전 호출자 확인 → 종료 시 **`410 Gone`** 반환 |

</div>

### 5.2 3대 무중단 배포 전략: 블루-그린 · 카나리 · 롤링

**서비스를 단 1초도 멈추지 않고 구버전(v1.0) 서버를 신버전(v1.1) 서버로 교체하는 세 가지 대표 방법은 '블루-그린', '카나리', '롤링'이며, 세 방법 모두 게이트웨이의 라우팅 비율(가중치)을 어떻게 바꾸느냐로 결정됩니다.**

#### 1) 블루-그린 배포 (Blue-Green Deployment): 두 벌을 준비해 한 번에 스위치 전환
구버전(**Blue**, 100%) 서버들이 일하는 옆에 신버전(**Green**, 0%) 서버 세트를 똑같은 대수로 미리 띄워 모델 로딩(Readiness)까지 끝내 놓습니다. 준비가 확인되면 게이트웨이의 라우팅 비율을 **`Blue 100 : Green 0`에서 `Blue 0 : Green 100`으로 단번에 전환**합니다.

![옛 버전 서버를 정지하지 않고 두었으므로 블루-그린의 전환과 롤백은 라우팅 테이블의 비율 한 줄로 끝납니다.](lms-img:0)

*옛 버전 서버를 정지하지 않고 두었으므로 블루-그린의 전환과 롤백은 라우팅 테이블의 비율 한 줄로 끝납니다.*

- **장점**: 구버전(Blue) 서버를 끄지 않고 잠시 대기시켜 두므로, 신버전에 문제가 발견되면 라우팅 비율만 다시 `100:0`으로 돌려 **1초 만에 즉시 롤백**할 수 있습니다.
- **단점**: 전환하는 동안 구버전과 신버전 서버를 동시에 띄워야 하므로 **여유 서버(GPU) 자원이 2배**로 필요합니다.

#### 2) 카나리 배포 (Canary Deployment): 소수(10%)에게 먼저 보내 검증하고 문제 시 자동 롤백
옛날 광부들이 탄광에 들어갈 때 유독가스에 민감한 카나리아 새를 먼저 들여보내 안전을 확인했던 것에서 유래한 이름입니다. 전체 트래픽을 한꺼번에 바꾸는 대신 **처음에는 10%만 신버전으로 보내 에러율(`5xx`)과 응답 시간을 지켜본 뒤, 안전하면 `30% → 50% → 100%`로 점진적으로 넓힙니다.** 만약 10% 구간에서 에러가 기준치를 넘으면 게이트웨이가 즉시 신버전 비중을 `0%`로 되돌려(**자동 롤백**) 피해 범위를 전체의 10% 이내로 가둡니다.

![같은 10%라도 요청마다 나누면 거의 모두가 두 버전을 번갈아 보고 사용자마다 나누면 일부만 새 버전을 계속 봅니다.](lms-img:0)

*같은 10%라도 요청마다 나누면 거의 모두가 두 버전을 번갈아 보고 사용자마다 나누면 일부만 새 버전을 계속 봅니다.*

- **주의할 점 (요청 기준 분할 vs 사용자 기준 분할)**: 위 그림처럼 멀티턴 대화가 이어지는 챗봇에서 단순히 **'매 요청마다 10% 제비뽑기'** 를 하면, 모든 직원이 10번 질문할 때 한두 번씩 신버전을 섞어 보게 되어 말투가 오락가락하고 오류도 전 직원이 겪게 됩니다. 따라서 대화형 서비스에서는 반드시 **'사용자(호출자 ID) 해시 기준'** 으로 10%의 사용자만 일관되게 신버전을 보도록 나누어야 합니다.

#### 3) 롤링 배포 (Rolling Deployment): 여유 서버가 없을 때 한 대씩 순차 교체
여유 GPU 서버가 없을 때 사용하는 기본 방식입니다. 기존 구버전 서버 3대 중 **1대를 먼저 라우팅에서 빼고(진행 중인 요청이 끝날 때까지 대기), 그 1대를 신버전으로 재시작해 준비가 끝나면 다시 투입**하는 과정을 한 대씩 차례로 반복합니다.

![롤링은 서버를 한 대씩 바꾸므로 여유 서버가 적게 들지만 바꾸는 동안 두 버전이 섞여 실행됩니다.](lms-img:0)

*롤링은 서버를 한 대씩 바꾸므로 여유 서버가 적게 들지만 바꾸는 동안 두 버전이 섞여 실행됩니다.*

<div style="font-size: 18px; line-height: 1.6;">

| 배포 전략 | 추가로 필요한 여유 서버(GPU) | 롤백(되돌리기) 속도 | 배포 중 두 버전 혼재 여부 | 적합한 상황 |
|---|---|---|---|---|
| **블루-그린 (Blue-Green)** | **전체 서버의 2배 (한 벌 더)** | **1초 미만 (라우팅만 원복)** | 전환 순간 외에는 섞이지 않음 | 여유 GPU가 충분하고 한 번에 깔끔히 전환·즉시 롤백해야 할 때 |
| **카나리 (Canary)** | 신버전 서버 1대 (소수) | **1초 미만 (가중치 0%로 변경)** | 검증 기간 동안 비율만큼 공존 | 새 모델의 실전 품질·안정성을 일부 트래픽(10%)으로 먼저 안전하게 검증할 때 |
| **롤링 (Rolling)** | **거의 없음 (0~1대)** | 느림 (다시 한 대씩 구버전으로 재기동) | 교체하는 내내 두 버전이 섞여 응답 | 여유 GPU가 없을 때 (실무에서는 **1대 카나리 검증 후 나머지 롤링 교체** 조합을 자주 사용) |

</div>

### 5.3 [기초 실습] 카나리 배포의 사용자 해시 분할과 자동 롤백 시뮬레이션

**라우팅 가중치(`{"v1.0": 90, "v1.1": 10}`)를 이용해 카나리 트래픽 분할과 에러 임계값 초과 시 자동 롤백이 얼마나 간단하게 동작하는지 파이썬 코드로 직접 확인합니다.**

아래 코드에서 `hashlib.md5(user_id.encode()).hexdigest()`는 사용자 ID 문자열을 고정 길이의 16진수 해시 문자열로 바꾸는 함수이며, 여기에 `int(..., 16) % 100`을 적용하면 16진수를 정수로 바꾼 뒤 100으로 나눈 나머지(`0~99` 버킷 번호)를 구해 **같은 사용자는 몇 번을 호출하든 항상 똑같은 버킷 번호**를 받게 합니다.

In [7]:
# [Step 1] 사용자 ID 해시(MD5) 기반 15% 카나리 분할 및 멀티턴 대화 일관성 확인
import hashlib

def route_by_user(user_id: str, canary_percent: int) -> str:
    """사용자 ID를 해시해 0~99 버킷으로 매핑함으로써 같은 사용자는 항상 같은 버전을 보게 합니다."""
    bucket = int(hashlib.md5(user_id.encode("utf-8")).hexdigest(), 16) % 100
    return "v1.1-canary" if bucket < canary_percent else "v1.0-stable"

# 20명의 사용자가 각각 5번씩 질문할 때, 대화 도중 두 버전을 섞어 본 사용자가 있는지 확인합니다
users = [f"user_{i:02d}" for i in range(1, 21)]
seen_versions = {u: {route_by_user(u, canary_percent=15) for _ in range(5)} for u in users}
mixed_users = [u for u, vers in seen_versions.items() if len(vers) > 1]
canary_users = [u for u, vers in seen_versions.items() if "v1.1-canary" in vers]

print(f"[15% 카나리 분할] 전체 20명 중 카나리 배정 사용자: {len(canary_users)}명 ({canary_users})")
print(f"[일관성 확인] 5턴 대화 도중 두 버전을 섞어 본 사용자 수: {len(mixed_users)}명 (0명이어야 정상)")

[15% 카나리 분할] 전체 20명 중 카나리 배정 사용자: 4명 (['user_02', 'user_04', 'user_14', 'user_19'])
[일관성 확인] 5턴 대화 도중 두 버전을 섞어 본 사용자 수: 0명 (0명이어야 정상)


In [8]:
# [Step 2] 카나리 자동 롤백(Auto Rollback) 감시: 5xx 에러가 3건 누적되면 즉시 카나리 가중치를 0%로 차단합니다
canary_weight = 15
error_count = 0
simulated_canary_statuses = [200, 500, 200, 500, 500, 200, 500]

print("[카나리 자동 롤백 감시 시뮬레이션]")
for idx, status in enumerate(simulated_canary_statuses, start=1):
    if status >= 500:
        error_count += 1
    print(f" - 카나리 요청 #{idx}: HTTP {status} (누적 5xx 에러: {error_count}건)")
    if error_count >= 3:
        canary_weight = 0
        print(f" -> [자동 롤백 발동] 5xx 에러 {error_count}건 도달! 카나리 가중치를 즉시 {canary_weight}%로 내리고 전량 v1.0-stable로 복구합니다.")
        break

[카나리 자동 롤백 감시 시뮬레이션]
 - 카나리 요청 #1: HTTP 200 (누적 5xx 에러: 0건)
 - 카나리 요청 #2: HTTP 500 (누적 5xx 에러: 1건)
 - 카나리 요청 #3: HTTP 200 (누적 5xx 에러: 1건)
 - 카나리 요청 #4: HTTP 500 (누적 5xx 에러: 2건)
 - 카나리 요청 #5: HTTP 500 (누적 5xx 에러: 3건)
 -> [자동 롤백 발동] 5xx 에러 3건 도달! 카나리 가중치를 즉시 0%로 내리고 전량 v1.0-stable로 복구합니다.


> **[문제] 사내 GPU 서버 4대가 1.0 버전으로 챗봇 서비스를 운영 중입니다.**
>
> 새로 파인튜닝한 1.1 버전 모델을 배포하려 하는데, 현재 남는 여유 GPU 서버가 한 대도 없고 사용자들은 멀티턴 대화를 이어가고 있습니다.
> 블루-그린, 카나리, 롤링 중 어떤 순서로 조합해 배포하겠습니까?
> - 먼저 기존 4대 중 1대를 라우팅에서 빼서 1.1 버전으로 재기동합니다.
> - 그 1대에 전체 트래픽의 **10~15%를 '사용자(호출자) 기준'**으로 배정(카나리)하고 자동 롤백 규칙을 걸어 둡니다.
> - 일정 시간 동안 에러율(`5xx`)과 응답 지연이 정상 범위임을 확인한 뒤, 나머지 3대를 한 대씩 순차적으로 1.1 버전으로 교체(롤링)합니다.

### 체크포인트

1. 모델 프롬프트만 다듬어 답변 품질을 개선했을 때와, 응답 JSON 필드 이름을 `response`에서 `answer`로 바꾸었을 때 각각 **서버 버전(`1.0.0` → `1.1.0`)** 과 **API 버전(`/v1` → `/v2`)** 중 무엇을 올려야 합니까? (목표 3)
2. 블루-그린 배포에서 롤백이 1초도 걸리지 않는 이유는 무엇이며, 어떤 조치를 취하면 그 즉시 롤백 이점이 사라집니까? (목표 3)
3. 멀티턴 챗봇을 카나리 배포할 때 '매 요청마다 10% 무작위 분할' 대신 '사용자(호출자) 기준 10% 분할'을 써야 하는 이유는 무엇입니까? (목표 3)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. 입출력 규격(약속)이 그대로인 품질 개선은 **서버 버전**만 올리면 되지만, 응답 필드 이름이 바뀌어 기존 클라이언트 코드가 깨지는 변경은 반드시 **새 API 버전(`/v2`)**을 열고 구버전(`/v1`)은 예고 기간(`Deprecation` 헤더)을 거친 뒤 닫아야 합니다.
2. 구버전(Blue) 서버들을 끄지 않고 그대로 대기시켜 두었기 때문에 라우팅 가중치만 되돌리면 즉시 롤백됩니다. 만약 전환 직후 **구버전 서버를 종료해 버리거나 구버전이 읽지 못하는 형태로 DB 스키마를 변경**하면 즉시 롤백 이점이 사라집니다.
3. 요청마다 무작위로 10%를 나누면 한 사용자의 대화 도중에 구버전과 신버전이 번갈아 응답해 대화 일관성이 깨지고, 신버전에 버그가 있을 때 거의 모든 사용자가 한 번씩 오류를 겪게 되기 때문입니다.

</details>

## 6. [51일차 미니 프로젝트 연결] 프로젝트 1: 정형 데이터 예측 서비스

## 6. [오후 종합 실습 & 프로젝트 연결] 5대 관문 통합 미니 게이트웨이 조립 및 프로젝트 1 안내

### 6.1 오늘 배운 5대 관문을 하나로 조립하기: `MiniAPIGateway`

**1~5장에서 개별적으로 구현한 ① `Trace ID` 발급, ② `401` API Key 인증, ③ `429` 호출자별 토큰 버킷, ④ `503` 대기열 상한 검사, ⑤ `Readiness` 필터링 및 최소 연결(`Least Connections`) 라우팅을 하나의 게이트웨이 파이프라인으로 조립해 5가지 실전 요청 시나리오를 검증합니다.**

In [9]:
class MiniAPIGateway:
    """교재 6.5절의 Fail-Fast 순서(Trace ID -> 401 -> 429 -> 503 -> Readiness + 최소연결)를 구현한 통합 게이트웨이입니다."""
    def __init__(self, valid_keys: dict, servers: list[dict], max_queue: int):
        self.valid_keys = valid_keys
        self.servers = servers
        self.max_queue = max_queue
        self.current_queue = 0
        self.buckets = {k: TokenBucket(capacity=2, refill_per_sec=1.0) for k in valid_keys}

    def handle(self, api_key: str | None, now_sec: float) -> dict:
        trace_id = f"req-{uuid.uuid4().hex[:8]}"

        # 1관문: API Key 인증 (401)
        if not api_key or api_key not in self.valid_keys:
            return {"status": 401, "trace_id": trace_id, "reason": "Unauthorized (유효하지 않은 API Key)"}

        # 2관문: 호출자별 토큰 버킷 요청 제한 (429)
        allowed, retry_after = self.buckets[api_key].allow_request(now_sec)
        if not allowed:
            return {"status": 429, "trace_id": trace_id, "retry_after": retry_after, "reason": "Too Many Requests"}

        # 3관문: Readiness('healthy')를 통과한 준비된 서버 필터링
        ready_pool = [s for s in self.servers if is_server_ready(s["http_status"], s["health_body"])]
        if not ready_pool or self.current_queue >= self.max_queue:
            return {"status": 503, "trace_id": trace_id, "retry_after": 2.0, "reason": "Service Unavailable (슬롯/대기열 포화)"}

        # 4관문: 준비된 서버 중 현재 활성 연결(active_requests)이 가장 적은 서버 선택 (Least Connections)
        target = min(ready_pool, key=lambda s: s["active_requests"])
        return {"status": 200, "trace_id": trace_id, "routed_to": target["name"], "caller": self.valid_keys[api_key]}

# 통합 게이트웨이 생성 및 4가지 대표 시나리오 검증
gw_servers = [
    {"name": "api-1 (바쁨)",   "http_status": 200, "health_body": {"status": "healthy"}, "active_requests": 2},
    {"name": "api-2 (한가함)", "http_status": 200, "health_body": {"status": "healthy"}, "active_requests": 0},
    {"name": "api-3 (로딩중)", "http_status": 200, "health_body": {"status": "loading"}, "active_requests": 0},
]
gw = MiniAPIGateway(valid_keys={"key-team-a": "A팀"}, servers=gw_servers, max_queue=4)

print("1) 무인증 요청       ->", gw.handle(api_key=None, now_sec=0.0))
print("2) 정상 요청 #1      ->", gw.handle(api_key="key-team-a", now_sec=0.0))
print("3) 정상 요청 #2      ->", gw.handle(api_key="key-team-a", now_sec=0.0))
print("4) 3연속 버스트(429) ->", gw.handle(api_key="key-team-a", now_sec=0.0))

1) 무인증 요청       -> {'status': 401, 'trace_id': 'req-193b0fb9', 'reason': 'Unauthorized (유효하지 않은 API Key)'}
2) 정상 요청 #1      -> {'status': 200, 'trace_id': 'req-1e16d467', 'routed_to': 'api-2 (한가함)', 'caller': 'A팀'}
3) 정상 요청 #2      -> {'status': 200, 'trace_id': 'req-07ae70d8', 'routed_to': 'api-2 (한가함)', 'caller': 'A팀'}
4) 3연속 버스트(429) -> {'status': 429, 'trace_id': 'req-6f83532a', 'retry_after': 1.0, 'reason': 'Too Many Requests'}


### 6.2 [51일차 오후 미니 프로젝트] 프로젝트 1: 정형 데이터 예측 서비스 (`D52_Appendix_Housing.ipynb`)

**게이트웨이와 무중단 배포의 기초 원리를 모두 익혔다면, 오후 시간에는 46~51일차의 서빙 파이프라인(직렬화 → FastAPI + Pydantic → 비동기 스레드풀 → Streamlit 대시보드 → 통합 테스트)을 내 손으로 처음부터 끝까지 완주하는 첫 번째 미니 프로젝트인 `D52_Appendix_Housing.ipynb`를 진행합니다.**

**지금까지 배운 모델 배포의 전체 흐름(직렬화 → FastAPI + Pydantic → 비동기 스레드풀 → Streamlit 대시보드 → 통합 테스트)을 내 손으로 처음부터 끝까지 완주해 보기 위해, 이어서 51일차 실습 부록 노트북인 `D52_Appendix_Housing.ipynb`(`[프로젝트 1] 정형 데이터 예측 서비스`)를 진행합니다.**

<div style="font-size: 18px; line-height: 1.6;">

| 단계 | `D52_Appendix_Housing.ipynb`에서 직접 구현할 내용 | 연계 일차 |
|---|---|---|
| **Step 1. 모델 및 전처리 저장** | 캘리포니아 주택 가격 회귀 모델(`HousingModel`) 학습 후 가중치(`state_dict`)와 정규화 통계(`mean`, `std`)를 분리 저장 | 46일차 |
| **Step 2. FastAPI 백엔드** | 8개 주택 특성 입력 범위 검증(`Pydantic Field`) + `run_in_executor` 추론 엔드포인트(`/predict`, `/health`) 구현 | 47 ~ 48일차 |
| **Step 3. Streamlit 대시보드** | 슬라이더·숫자 입력 폼으로 FastAPI 서버를 호출하고 예측 집값과 레이더 차트를 시각화하는 UI 구현 | 49일차 |
| **Step 4. 통합 테스트** | 정상 요청(`200`), 범위 초과·필수값 누락(`422`), 동시 요청 처리 여부를 코드로 일괄 검증 | 50 ~ 51일차 |

</div>

## 오늘 정리

- **1장 (API 게이트웨이의 역할과 빠른 거절)**: 서버가 여러 대가 되면 앞단에 단일 진입점인 게이트웨이를 두어 라우팅, 부하 분산(라운드로빈·최소 연결), 인증, 요청 제한, 로깅을 일원화합니다. 게이트웨이는 비용이 가장 적게 드는 확인(`401 인증 → 429 한도 → 503 대기열`)부터 먼저 수행해 GPU 자원을 아낍니다.
- **2장 (통합 인증과 `Trace ID`)**: 백엔드 서버 포트는 내부망으로 닫아 우회 접근(뒷문)을 차단하고, 게이트웨이가 진입 순간에 `Trace ID`(`X-Request-ID`)를 부여해 다중 서버 로그를 하나로 연결합니다.
- **3장 (요청 제한 `429`와 대기열 `503`)**: 토큰 버킷 알고리즘으로 호출자별 평균 속도와 순간 버스트를 제어하며, 개별 호출자 한도 초과는 `429`, 서버 전체 슬롯·대기열 포화는 `503`에 `Retry-After` 헤더를 담아 반환합니다.
- **4장 (생존 확인과 준비 확인)**: 프로세스가 살아 있는 상태(Liveness)와 모델 로딩이 끝난 상태(Readiness)를 분리해 모델 로딩 중인 서버에는 요청을 보내지 않으며, `/health` 안에는 실제 모델 생성이나 순간 부하를 넣지 않습니다.
- **5장 (API 수명 주기와 3대 무중단 배포)**: 입출력 약속이 바뀌면 새 API 버전(`/v2`)을 열고 예고 후 폐기(`410`)합니다. 여유 자원 유무와 서비스 특성에 맞춰 **블루-그린**(2배 자원, 즉시 전환·롤백), **카나리**(소수 비율 선검증 및 자동 롤백), **롤링**(순차 교체) 배포를 선택합니다.

### 제출

다음 항목을 정리해 제출하십시오.

1. 3.2절 토큰 버킷 시뮬레이션 실행 결과와 3.3절 대기열 길이 계산 결과
2. 4.2절 헬스체크 Readiness 판정 실행 결과 및 5.3절 카나리 자동 롤백 시뮬레이션 출력
3. 이어서 진행한 `[51일차 실습 부록 · 프로젝트 1] 정형 데이터 예측 서비스`(`D52_Appendix_Housing.ipynb`)의 통합 테스트 결과
4. 각 장 체크포인트 문항의 답변

## Reference

**교재**  
- 아비 아리안, 『LLMOps 완벽 가이드』, 박조은·박주환 옮김, 한빛미디어, 2026 (6.1.5 5단계: 모니터링 설정하기, 6.4 자격 증명 관리, 6.5 API 게이트웨이, 6.6 API 버전 관리 및 수명 주기 관리, 6.7 LLM 배포 아키텍처)

**공식 규격 및 기술 문서**  
- IETF, [RFC 9110: HTTP Semantics](https://www.rfc-editor.org/rfc/rfc9110.html) & [RFC 6585: Additional HTTP Status Codes (429)](https://www.rfc-editor.org/rfc/rfc6585.html)  
- Kubernetes Docs, [Liveness, Readiness, and Startup Probes](https://kubernetes.io/docs/concepts/workloads/pods/probes/)  
- Google SRE Workbook, [Canarying Releases](https://sre.google/workbook/canarying-releases/)